In [1]:
# NLP + K-MEANS PATIENT DATASET

import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

# 1. Load dataset
df = pd.read_csv("dataset_pasien_kompleks.csv")
print(df.head())
print(df.columns)

# 2. Select text column
# Change this to your actual text column
text_col = "keluhan"

df[text_col] = df[text_col].fillna("").astype(str)

# 3. Clean text
def clean_text(x):
    x = x.lower()
    x = re.sub(r"[^a-zA-Z\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

df["text_clean"] = df[text_col].apply(clean_text)

# 4. TF-IDF
tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2)
)
X = tfidf.fit_transform(df["text_clean"])

# 5. K-Means
k = 4
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# 6. Show results
print("\nPatients per cluster:")
print(df["cluster"].value_counts().sort_index())

print("\nTop words per cluster:")
words = tfidf.get_feature_names_out()

for i, center in enumerate(model.cluster_centers_):
    top = center.argsort()[-10:][::-1]
    print(f"Cluster {i}:",
          ", ".join(words[top]))

# 7. Visualize
svd = TruncatedSVD(n_components=2, random_state=42)
X_2D = svd.fit_transform(X)

plt.figure(figsize=(8, 6))
plt.scatter(X_2D[:, 0], X_2D[:, 1], c=df["cluster"], alpha=0.7)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Patient Clusters - NLP + K-Means")
plt.colorbar(label="Cluster")
plt.show()

# 8. Save result
df.to_csv("hasil_kmeans_pasien.csv", index=False)
print("\nSaved: hasil_kmeans_pasien.csv")

  ID_Pasien                   Nama               NIK Tanggal_Lahir  \
0   PSN1000  Devi Oktaviani, M.TI.  5086369621880525    2010-10-11   
1   PSN1001           Umay Hariyah  9473882799712303    2000-02-11   
2   PSN1002      Fitriani Najmudin  8630345434677077    1941-12-13   
3   PSN1003        Fitriani Rahayu  8666617742004210    1948-10-16   
4   PSN1004      Yusuf Pudjiastuti  8287655072244799    2011-10-28   

                                              Alamat               Kota  \
0          Jl. Ciwastra No. 02, Banjarbaru, MU 43082        Lhokseumawe   
1          Jalan BKR No. 99, Manado, Gorontalo 96361            Sibolga   
2   Gang Joyoboyo No. 311, Banjarbaru, Lampung 05065  Tangerang Selatan   
3  Jalan Moch. Ramdan No. 472, Surakarta, Nusa Te...        Bukittinggi   
4     Jalan Rumah Sakit No. 04, Purwokerto, SS 25970         Banda Aceh   

           Provinsi              No_HP                      Email  Penyakit  \
0  Kalimantan Utara       082 099 6043  candrapus

KeyError: 'keluhan'